# BusDemand AI – Model Development & Evaluation (Stage 7)
## Baseline vs Random Forest vs HistGradientBoosting

This notebook trains and compares three forecasting models on the 87,680 synthetic trip records:
1. **Historical Baseline:** Route x Slot historical averages with hierarchical fallback.
2. **Random Forest Regressor:** Scikit-learn Pipeline with One-Hot encoding.
3. **HistGradientBoosting Regressor:** Scikit-learn Pipeline with native categorical handling.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import json

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 5)

# Load evaluation metrics summary
with open('../artifacts/evaluation_metrics.json') as f:
    metrics = json.load(f)

val_df = pd.DataFrame(metrics['validation_metrics'])
test_df = pd.DataFrame(metrics['test_metrics'])
print('Validation Comparison:')
print(val_df)
print('\nTest Comparison:')
print(test_df)

### 1. Model Comparison on Validation Set (MAE & WAPE)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# MAE Bar Chart
bars1 = axes[0].bar(val_df['model'], val_df['mae'], color=['#7f7f7f', '#2ca02c', '#1f77b4'], edgecolor='black')
axes[0].set_title('Validation MAE (Lower is Better)')
axes[0].set_ylabel('Mean Absolute Error (passengers)')
for bar in bars1:
    yval = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2.0, yval + 0.1, f'{yval:.2f}', ha='center', va='bottom', fontweight='bold')

# WAPE Bar Chart
bars2 = axes[1].bar(val_df['model'], val_df['wape_pct'], color=['#7f7f7f', '#2ca02c', '#1f77b4'], edgecolor='black')
axes[1].set_title('Validation WAPE % (Lower is Better)')
axes[1].set_ylabel('Weighted Absolute Percentage Error (%)')
for bar in bars2:
    yval = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2.0, yval + 0.3, f'{yval:.1f}%', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

### 2. Predictions vs Actuals on Held-out Test Set

In [ ]:
# Load best model and generate test predictions
best_model = joblib.load('../artifacts/best_model_pipeline.joblib')
df = pd.read_csv('../data/processed/trip_demand.csv')
test_df_full = df[df['departure_date'] >= '2026-07-01'].copy()

from src.models.preprocessing import MODEL_FEATURE_COLUMNS
test_preds = best_model.predict(test_df_full[MODEL_FEATURE_COLUMNS])
test_df_full['pred'] = test_preds

plt.figure(figsize=(8, 8))
plt.scatter(test_df_full['passenger_demand'], test_df_full['pred'], alpha=0.15, color='#1f77b4', s=10)
plt.plot([0, 160], [0, 160], 'r--', linewidth=2, label='Perfect 1:1 Line')
plt.xlabel('Actual Passenger Demand')
plt.ylabel('Predicted Passenger Demand')
plt.title(f'Actual vs Predicted Demand on Held-Out Test Set ({metrics["best_model"]})')
plt.legend()
plt.tight_layout()
plt.show()